In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [17]:
import os, random, math, re
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.preprocessing import normalize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.metrics import ndcg_score
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

In [18]:
# =============================
# 1) 경로 설정 (환경에 맞게 수정)
# =============================
MOVIES_CSV = "/content/drive/MyDrive/bigData/data/movie_dataset/movies.csv"
GAMES_CSV  = "/content/drive/MyDrive/bigData/data/game_dataset/cleaned_games.csv"
MOVIE_EMB  = "/content/drive/MyDrive/bigData/hybrid_movie_embeddings.npy"
GAME_EMB   = "/content/drive/MyDrive/bigData/hybrid_game_embeddings_BERT_final.npy"
RATINGS_CSV = "/content/drive/MyDrive/bigData/data/movie_dataset/ratings.csv"  # 사용자 평점 (userId, movieId, rating, movie_index 등)
MODEL_SAVE = "models/mapper_genre_tag.pth"

# =============================
# 2) 환경/장치
# =============================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# =============================
# 3) 데이터 로드
# =============================
movies = pd.read_csv(MOVIES_CSV)
games  = pd.read_csv(GAMES_CSV)
movie_embs = np.load(MOVIE_EMB)
game_embs  = np.load(GAME_EMB)
ratings = pd.read_csv(RATINGS_CSV)


Device: cpu


In [46]:
# =============================
# 4) 기본 전처리: 장르/태그 컬럼 정리 & L2 정규화
# =============================
import ast

def parse_movie_genres(s):
    try:
        return [g.strip().lower() for g in str(s).split("|") if g and g != "(no genres listed)"]
    except:
        return []

def parse_game_genres(s):
    if pd.isna(s):
        return []
    s = str(s).strip()
    # case 1: looks like Python list (starts with [ and ends with ])
    if s.startswith("[") and s.endswith("]"):
        try:
            parsed = ast.literal_eval(s)
            # ensure it's list
            if isinstance(parsed, list):
                return [g.strip().lower() for g in parsed if isinstance(g, str)]
        except Exception:
            pass
    # fallback: split by common delimiters
    for sep in ["|", ",", ";"]:
        if sep in s:
            return [g.strip().lower() for g in s.split(sep) if g.strip()]
    # otherwise single token
    return [s.lower()]

movies["genre_list"] = movies.get("genres", "").apply(parse_movie_genres)
games["genre_list"]  = games.get("genres", "").apply(parse_game_genres)


# tag_text 컬럼 준비 (tags 존재하면 사용, 없으면 overview/title)
movies["tag_text"] = movies.get("tags", movies.get("overview", movies.get("title", ""))).fillna("").astype(str)
games["tag_text"]  = games.get("tags",  games.get("description", games.get("title", ""))).fillna("").astype(str)

# L2 정규화
movie_embs = normalize(movie_embs, axis=1)
game_embs  = normalize(game_embs, axis=1)

print("movie_embs.shape:", movie_embs.shape, "game_embs.shape:", game_embs.shape)

# =============================
# 5) 장르 매핑 테이블 (영화 장르 -> 게임 장르 후보)
#    필요하면 도메인에 맞게 이 dict를 수정하세요.
# =============================
genre_mapping = {
    "action": ["action", "shooter", "fighting", "platformer"],
    "adventure": ["adventure", "rpg", "puzzle", "exploration"],
    "comedy": ["casual", "party", "simulation"],
    "drama": ["visual novel", "narrative", "interactive story"],
    "fantasy": ["fantasy", "rpg", "adventure"],
    "sci-fi": ["sci-fi", "space", "strategy"],
    "horror": ["horror", "survival", "thriller"],
    "romance": ["visual novel", "dating sim", "simulation"],
    "thriller": ["horror", "survival", "stealth"],
    "mystery": ["adventure", "puzzle", "investigation"],
    "biography": ["visual novel", "narrative"],
    "crime": ["detective", "investigation", "thriller"],
    "documentary": ["simulation", "education"],
    "music": ["rhythm", "casual"],
    "war": ["strategy", "simulation", "tactical"],
    "western": ["action", "shooter"],
    "animation": ["casual", "visual novel"],
}

def movie_genres_to_game_genres(movie_genre_list, mapping):
    import re
    mapped = set()
    for mg in movie_genre_list:
        mg_norm = mg.lower().strip().replace("-", " ").replace("/", " ")
        for key in mapping.keys():
            key_norm = key.lower().strip().replace("-", " ")
            if re.search(rf"\b{re.escape(key_norm)}\b", mg_norm) or key_norm in mg_norm:
                for gg in mapping[key]:
                    mapped.add(gg.lower().strip())
    return mapped



movie_embs.shape: (9742, 1723) game_embs.shape: (5000, 789)


In [32]:
# =============================
# 6) TF-IDF 태그 벡터화 (movies+games 통합 corpus)
# =============================
corpus_movies = movies["tag_text"].fillna("").astype(str).tolist()
corpus_games  = games["tag_text"].fillna("").astype(str).tolist()
corpus_all = corpus_movies + corpus_games

tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1,2))
tfidf_all = tfidf.fit_transform(corpus_all)
movie_tag_vecs = tfidf_all[:len(corpus_movies)]
game_tag_vecs  = tfidf_all[len(corpus_movies):]

# =============================
# 7) Candidate pair 생성 (장르-mapping 기반 -> 태그 유사도 필터)
# =============================
from collections import defaultdict
game_genre_index = defaultdict(list)
for gi, g_row in games.iterrows():
    for gg in g_row["genre_list"]:
        game_genre_index[gg.lower().strip()].append(gi)

pairs = []
THRESH_TAG_SIM = 0.05

print("Generating candidate pairs...")
for mi, m_row in tqdm(movies.iterrows(), total=len(movies)):
    mapped_game_genres = movie_genres_to_game_genres(m_row["genre_list"], genre_mapping)
    if not mapped_game_genres:
        continue
    candidate_set = set()
    for mg in mapped_game_genres:
        candidate_set.update(game_genre_index.get(mg, []))
    if not candidate_set:
        continue
    cand_list = list(candidate_set)
    # compute tag sim for candidates
    movie_vec = movie_tag_vecs[mi]
    game_slice = game_tag_vecs[cand_list]
    sims = cosine_similarity(movie_vec, game_slice).flatten()
    for idx_cand, sim_score in zip(cand_list, sims):
        if sim_score >= THRESH_TAG_SIM:
            # genre overlap count (movie genre in game's genre list)
            overlap_count = 0
            for mg in m_row["genre_list"]:
                if mg.lower().strip() in games.loc[idx_cand, "genre_list"]:
                    overlap_count += 1
            genre_score = overlap_count
            combined = 0.7 * (genre_score) + 0.3 * sim_score
            pairs.append((mi, idx_cand, combined))

print("Total candidate pairs:", len(pairs))
# sort and trim
pairs = sorted(pairs, key=lambda x: x[2], reverse=True)
MAX_PAIRS = 200000
pairs = pairs[:MAX_PAIRS]
print("Using pairs:", len(pairs))

# =============================
# 8) train/val split for pairs
# =============================
train_pairs, val_pairs = train_test_split(pairs, test_size=0.1, random_state=42)
print("Train pairs:", len(train_pairs), "Val pairs:", len(val_pairs))

# =============================
# 9) Dataset / DataLoader
# =============================
class MovieGamePairDataset(Dataset):
    def __init__(self, pairs, movie_embs, game_embs):
        self.pairs = pairs
        self.movie_embs = movie_embs
        self.game_embs = game_embs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        mi, gi, w = self.pairs[idx]
        mv = self.movie_embs[mi]
        gv = self.game_embs[gi]
        return torch.from_numpy(mv).float(), torch.from_numpy(gv).float(), torch.tensor(float(w), dtype=torch.float32)

BATCH_SIZE = 128
train_loader = DataLoader(MovieGamePairDataset(train_pairs, movie_embs, game_embs), batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader   = DataLoader(MovieGamePairDataset(val_pairs, movie_embs, game_embs), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)


Generating candidate pairs...


100%|██████████| 9742/9742 [00:27<00:00, 349.70it/s]


Total candidate pairs: 129486
Using pairs: 129486
Train pairs: 116537 Val pairs: 12949


In [33]:
# =============================
# 10) Mapper 모델
# =============================
class MovieToGameMapper(nn.Module):
    def __init__(self, input_dim, hidden_dim=512, output_dim=None, dropout=0.2):
        super().__init__()
        if output_dim is None:
            output_dim = input_dim
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, output_dim)
        )
    def forward(self, x):
        out = self.net(x)
        out = F.normalize(out, dim=1)  # L2 normalize output
        return out

input_dim = movie_embs.shape[1]
output_dim = game_embs.shape[1]
model = MovieToGameMapper(input_dim=input_dim, hidden_dim=512, output_dim=output_dim, dropout=0.2).to(device)

# =============================
# 11) Loss / Optimizer / Train loop (cosine loss weighted)
# =============================
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

def cosine_loss_weighted(pred, target, weight=None):
    # pred, target are expected to be normalized
    cos = (pred * target).sum(dim=1)  # (B,)
    loss_sample = 1.0 - cos
    if weight is not None:
        w = weight / (weight.mean() + 1e-8)
        return (loss_sample * w).mean()
    else:
        return loss_sample.mean()

EPOCHS = 12
for epoch in range(EPOCHS):
    model.train()
    running = 0.0
    for mb, gb, w in tqdm(train_loader, desc=f"Train Ep{epoch+1}/{EPOCHS}"):
        mb = mb.to(device); gb = gb.to(device); w = w.to(device)
        optimizer.zero_grad()
        out = model(mb)
        gb_n = F.normalize(gb, dim=1)
        loss = cosine_loss_weighted(out, gb_n, weight=w)
        loss.backward()
        optimizer.step()
        running += loss.item() * mb.size(0)
    train_avg = running / len(train_loader.dataset)
    # validation loss
    model.eval()
    val_running = 0.0
    with torch.no_grad():
        for mb, gb, w in val_loader:
            mb = mb.to(device); gb = gb.to(device); w = w.to(device)
            out = model(mb)
            loss_val = cosine_loss_weighted(out, F.normalize(gb, dim=1), weight=w)
            val_running += loss_val.item() * mb.size(0)
    val_avg = val_running / len(val_loader.dataset)
    print(f"[Epoch {epoch+1}] train_loss: {train_avg:.6f}  val_loss: {val_avg:.6f}")

# save model
os.makedirs("models", exist_ok=True)
torch.save(model.state_dict(), MODEL_SAVE)
print("Saved mapper ->", MODEL_SAVE)

# =============================
# 12) helper: map batch (numpy)
# =============================
def map_batch_np(X_np, model, device, batch_size=1024):
    model.eval()
    out_list = []
    for i in range(0, X_np.shape[0], batch_size):
        xb = torch.from_numpy(X_np[i:i+batch_size]).float().to(device)
        with torch.no_grad():
            out = model(xb).cpu().numpy()
        out_list.append(out)
    return np.vstack(out_list)


Train Ep1/12: 100%|██████████| 911/911 [00:31<00:00, 29.20it/s]


[Epoch 1] train_loss: 0.104292  val_loss: 0.093767


Train Ep2/12: 100%|██████████| 911/911 [01:01<00:00, 14.93it/s]


[Epoch 2] train_loss: 0.093756  val_loss: 0.093550


Train Ep3/12: 100%|██████████| 911/911 [01:38<00:00,  9.28it/s]


...

[Epoch 10] train_loss: 0.093215  val_loss: 0.093169


Train Ep11/12: 100%|██████████| 911/911 [05:05<00:00,  2.98it/s]


[Epoch 11] train_loss: 0.093215  val_loss: 0.093245


Train Ep12/12: 100%|██████████| 911/911 [05:36<00:00,  2.71it/s]


[Epoch 12] train_loss: 0.093142  val_loss: 0.093100
Saved mapper -> models/mapper_genre_tag.pth


In [49]:
# =============================
# 13) 사용자 벡터 생성: 평점 가중 평균
#    전제: ratings.csv 에 userId, movieId (또는 movie_index), rating 컬럼 존재
#    만약 movie_index 가 없다면 movies dataframe에서 movieId->index 매핑 필요
# =============================
# build movieId -> movie_index mapping if needed
if "movie_index" in ratings.columns:
    # assume this index corresponds to movie_embs index
    pass
else:
    # try map by movieId -> movies.index
    if "movieId" in movies.columns and "movieId" in ratings.columns:
        id2idx = {mid: idx for idx, mid in movies["movieId"].items()}
        ratings["movie_index"] = ratings["movieId"].map(id2idx)
    else:
        raise RuntimeError("ratings.csv must contain movieId or movie_index to map to embeddings.")

def user_weighted_vector(user_id, ratings_df, movie_embs, min_rating=3.5):
    user_df = ratings_df[ratings_df["userId"] == user_id]
    liked = user_df[user_df["rating"] >= min_rating]
    if liked.empty:
        return None
    # drop missing movie_index
    liked = liked.dropna(subset=["movie_index"])
    idxs = liked["movie_index"].astype(int).values
    scores = liked["rating"].values.astype(float)
    embs = movie_embs[idxs]
    weighted = (embs * scores.reshape(-1,1)).sum(axis=0) / (scores.sum() + 1e-8)
    # normalize
    weighted = weighted / (np.linalg.norm(weighted) + 1e-8)
    return weighted

# =============================
# 14) 추천 함수: user->mapped->similarity->TopK
# =============================
def recommend_with_mapper(user_id, ratings_df, movie_embs, game_embs, model, top_k, min_rating=3.5):
    uvec = user_weighted_vector(user_id, ratings_df, movie_embs, min_rating=min_rating)
    if uvec is None:
        return []
    # map through model
    mapped = map_batch_np(uvec.reshape(1,-1), model, device)  # normalized
    # ensure game_embs normalized
    game_norm = normalize(game_embs, axis=1)
    sims = cosine_similarity(mapped, game_norm).flatten()
    top_idx = sims.argsort()[::-1][:top_k]
    return [(int(i), float(sims[i])) for i in top_idx]


In [57]:
# =============================
# 15) 추천 예시 (유저 하나)
# =============================
sample_user = ratings["userId"].unique()[0]
print("Sample user:", sample_user)
recs = recommend_with_mapper(sample_user, ratings, movie_embs, game_embs, model, top_k=5, min_rating=4.0)
print("Top recommendations (game_idx, score):", recs)
print("추천 게임 제목 리스트:")
for gi, sc in recs:
    print(games.iloc[gi]["title"])

# =============================
# 16) 평가 루틴: Precision@K, Recall@K, NDCG@K for users
#    - ground truth: 사용자가 테스트셋에서 rating >= threshold 로 좋아한 게임들
#    - NOTE: 이 데이터셋에는 사용자-게임 상호작용이 없으므로 우리는 "proxy" 평가 사용:
#      validation pairs 포함된 영화->게임 대응을 이용한 recall@K (mapping 품질)
#    - 추가: 실제 게임 좋아요 데이터가 있다면 사용자 기반 평가로 바꾸세요.
# =============================
# Evaluate mapping quality on val_pairs (recall@K) - already computed below as sanity check
#VAL_N = min(2000, len(val_pairs))
#val_pairs_small = val_pairs[:VAL_N]
#val_movie_idx = [p[0] for p in val_pairs_small]
#val_game_idx  = [p[1] for p in val_pairs_small]
#val_movie_embs = movie_embs[val_movie_idx]
#mapped_val = map_batch_np(val_movie_embs, model, device, batch_size=512)
#game_embs_norm = normalize(game_embs, axis=1)
#sims_mat = cosine_similarity(mapped_val, game_embs_norm)
#K_list = [5,10,20]
#for K in K_list:
    #hits = 0
    #for i, true_g in enumerate(val_game_idx):
        #topk = sims_mat[i].argsort()[::-1][:K]
        #if true_g in topk:
            #hits += 1
    #recall = hits / len(val_pairs_small)
    #print(f"Recall@{K}: {recall:.4f}")

# =============================
# 17) (선택) 전체 사용자 평가 예시 (모집단 모집한다면)
#    - 사용자가 영화 평점을 줬을 때(테스트셋) 게임 추천의 정합도를 평가하려면
#      실제 사용자-게임 상호작용(예: play_count, purchase) 이 필요.
#    - 여기서는 예시 코드(데이터 있을 때 사용)
# =============================
# def evaluate_users(ratings_df, users_list, top_k=10, min_rating=4.0):
#     metrics = {"precision":[], "recall":[], "ndcg":[]}
#     for uid in users_list:
#         # 실제로 사용자가 좋아한 게임(ground truth) 로딩 필요
#         # ground_truth_games = ...
#         # recs = recommend_with_mapper(uid, ratings_df, movie_embs, game_embs, model, top_k=top_k, min_rating=min_rating)
#         # rec_idx = [i for i,_ in recs]
#         # compute precision/recall/ndcg...
#         pass

print("\nnotebook finished. 모델 경로:", MODEL_SAVE)

Sample user: 1
Top recommendations (game_idx, score): [(1486, 0.9647307134714797), (3340, 0.9641383992014696), (1108, 0.9628010440822236), (2586, 0.9625887921580765), (4315, 0.9621311144833964)]
추천 게임 제목 리스트:
Time Gentlemen, Please! and Ben There, Dan That! Special Edition Double Pack
Struggling
ELEX
Heaven's Vault
Survivalist

notebook finished. 모델 경로: models/mapper_genre_tag.pth
